# Warehouse Allocation

## Objective

This notebook assigns the nearest disaster response warehouse to every disaster
event.

Each disaster event is matched with the closest warehouse using geographic
coordinates.

The assigned warehouse will later provide

- Food
- Water
- Medical Kits
- Blankets
- Tents
- Rescue Teams
- Vehicles

for disaster response.

---

### Input

datasets/affected_population.csv

datasets/inventory.csv

---

### Output

datasets/disaster_with_warehouse.csv

---

### Pipeline

Affected Population Dataset

+

Warehouse Inventory

↓

Nearest Warehouse Search

↓

Assigned Warehouse

↓

disaster_with_warehouse.csv

In [42]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree

from math import radians, sin, cos, sqrt, atan2

In [43]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_DIR = Path.cwd().parents[1]

DATA_DIR = PROJECT_DIR / "datasets"

EVENTS_FILE = DATA_DIR / "affected_population.csv"

INVENTORY_FILE = DATA_DIR / "inventory.csv"

OUTPUT_FILE = DATA_DIR / "disaster_with_warehouse.csv"

In [44]:
assert EVENTS_FILE.exists()

assert INVENTORY_FILE.exists()

print("Datasets Found")

Datasets Found


In [45]:
events = pd.read_csv(EVENTS_FILE)

inventory = pd.read_csv(INVENTORY_FILE)

print(events.shape)

print(inventory.shape)

(2374, 32)
(160, 7)


In [46]:
warehouses = (

    inventory[

        [

            "WarehouseID",

            "WarehouseName",

            "State",

            "Latitude",

            "Longitude"

        ]

    ]

    .drop_duplicates()

    .reset_index(drop=True)

)

warehouses

,WarehouseID,WarehouseName,State,Latitude,Longitude
0,WH001,Srinagar Warehouse,Jammu and Kashmir,34.0837,74.7973
1,WH002,Chandigarh Warehouse,Chandigarh,30.7333,76.7794
2,WH003,Delhi Central Warehouse,Delhi,28.6139,77.2090
3,WH004,Lucknow Warehouse,Uttar Pradesh,26.8467,80.9462
4,WH005,Jaipur Warehouse,Rajasthan,26.9124,75.7873
5,WH006,Ahmedabad Warehouse,Gujarat,23.0225,72.5714
6,WH007,Mumbai Warehouse,Maharashtra,19.0760,72.8777
7,WH008,Bhopal Warehouse,Madhya Pradesh,23.2599,77.4126
8,WH009,Raipur Warehouse,Chhattisgarh,21.2514,81.6296
9,WH010,Bhubaneswar Warehouse,Odisha,20.2961,85.8245


In [47]:
warehouse_tree = cKDTree(

    np.column_stack(

        [

            warehouses["Longitude"],

            warehouses["Latitude"]

        ]

    )

)

print("Warehouse KDTree Created")

Warehouse KDTree Created


In [48]:
distances, indices = warehouse_tree.query(

    np.column_stack(

        [

            events["Longitude"],

            events["Latitude"]

        ]

    )

)

In [49]:
# ============================================================
# Haversine Distance Function
# ============================================================

EARTH_RADIUS_KM = 6371.0


def haversine_distance(lat1, lon1, lat2, lon2):

    lat1 = radians(lat1)
    lon1 = radians(lon1)

    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        +
        cos(lat1)
        *
        cos(lat2)
        *
        sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return EARTH_RADIUS_KM * c

In [50]:
events["AssignedWarehouseID"] = (

    warehouses.iloc[indices]

    ["WarehouseID"]

    .values

)

events["AssignedWarehouse"] = (

    warehouses.iloc[indices]

    ["WarehouseName"]

    .values

)

events["WarehouseState"] = (

    warehouses.iloc[indices]

    ["State"]

    .values

)

In [51]:
# ============================================================
# Compute Accurate Haversine Distance
# ============================================================

warehouse_latitudes = (

    warehouses.iloc[indices]["Latitude"]

    .values

)

warehouse_longitudes = (

    warehouses.iloc[indices]["Longitude"]

    .values

)

events["DistanceToWarehouseKm"] = [

    haversine_distance(

        event_lat,

        event_lon,

        warehouse_lat,

        warehouse_lon

    )

    for event_lat, event_lon, warehouse_lat, warehouse_lon in zip(

        events["Latitude"],

        events["Longitude"],

        warehouse_latitudes,

        warehouse_longitudes

    )

]

events["DistanceToWarehouseKm"] = (

    events["DistanceToWarehouseKm"]

    .round(2)

)

In [52]:
events[

    [

        "Latitude",

        "Longitude",

        "AssignedWarehouse",

        "DistanceToWarehouseKm"

    ]

].head()

,Latitude,Longitude,AssignedWarehouse,DistanceToWarehouseKm
0,27.5051,92.1524,Guwahati Warehouse,156.83
1,27.4598,92.0424,Guwahati Warehouse,149.38
2,24.3478,94.0847,Guwahati Warehouse,309.34
3,24.9682,93.0641,Guwahati Warehouse,186.69
4,32.3888,76.6062,Chandigarh Warehouse,184.81


In [53]:
print()

print(

    events["AssignedWarehouse"]

    .value_counts()

)

print()

print(

    events["DistanceToWarehouseKm"]

    .describe()

)


AssignedWarehouse
Guwahati Warehouse         759
Port Blair Warehouse       727
Srinagar Warehouse         373
Ahmedabad Warehouse        179
Chandigarh Warehouse       148
Delhi Central Warehouse     50
Lucknow Warehouse           46
Mumbai Warehouse            35
Kolkata Warehouse           20
Raipur Warehouse            12
Jaipur Warehouse            11
Hyderabad Warehouse         10
Bhopal Warehouse             2
Bhubaneswar Warehouse        2
Name: count, dtype: int64

count    2374.000000
mean      233.848020
std       126.222679
min         0.240000
25%       131.110000
50%       219.625000
75%       331.387500
max       608.250000
Name: DistanceToWarehouseKm, dtype: float64


In [54]:
events.to_csv(

    OUTPUT_FILE,

    index=False

)

print()

print("Saved Successfully")

print()

print(OUTPUT_FILE)


Saved Successfully

C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\disaster_with_warehouse.csv


In [55]:
loaded = pd.read_csv(

    OUTPUT_FILE

)

loaded.head()

,SourceEventID,DisasterType,Latitude,Longitude,Magnitude,Depth,Time,Temperature,Humidity,Rainfall,...,ExposureRatio,AffectedPopulation,Children,Adults,Elderly,VulnerabilityScore,AssignedWarehouseID,AssignedWarehouse,WarehouseState,DistanceToWarehouseKm
0,18,Earthquake,27.5051,92.1524,4.3,10.000,2026-06-21 00:53:20.927000,0.526455,0.831579,0.0,...,0.183250,3374,911,2024,439,1.12,WH012,Guwahati Warehouse,Assam,156.83
1,19,Earthquake,27.4598,92.0424,4.3,10.000,2026-06-21 00:52:54.610000,0.584656,0.915789,0.0,...,0.186286,5429,1466,3257,706,1.12,WH012,Guwahati Warehouse,Assam,149.38
2,22,Earthquake,24.3478,94.0847,4.5,52.125,2026-06-18 15:29:24.907000,0.718254,0.915789,0.0,...,0.223250,68999,18630,41399,8970,1.12,WH012,Guwahati Warehouse,Assam,309.34
3,39,Earthquake,24.9682,93.0641,4.4,48.154,2026-06-11 15:40:35.593000,0.805556,0.863158,0.0,...,0.221373,99816,26950,59890,12976,1.12,WH012,Guwahati Warehouse,Assam,186.69
4,46,Earthquake,32.3888,76.6062,4.9,10.000,2026-06-05 16:34:36.131000,0.395503,0.978947,0.0,...,0.206315,4794,1294,2876,623,1.12,WH002,Chandigarh Warehouse,Chandigarh,184.81
